# HyDE
For a given query, HyDE retrieval pipeline contains 4 components:
1. Promptor: bulid prompt for generator based on specific task.
2. Generator: generates hypothesis documents using Large Language Model.
3. Encoder: encode hypothesis documents to HyDE vector.
4. Searcher: search nearest neighbour for the HyDE vector (dense retrieval).

### Initialize HyDE components
We use [pyserini](https://github.com/castorini/pyserini) as the search interface.

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"

# %pip install -e .
import json
from pyserini.search.faiss import FaissSearcher
from pyserini.search.lucene import LuceneSearcher
from pyserini.encode import AutoQueryEncoder

from hyde import Promptor, OpenAIGenerator, CohereGenerator, HyDE

In [ ]:
KEY = 'YOUR_API_KEY'
promptor = Promptor('web search')
generator = OpenAIGenerator('gpt-4o-mini', KEY)
encoder = AutoQueryEncoder(encoder_dir='facebook/contriever', pooling='mean', use_fp16=False)
searcher = FaissSearcher('contriever_msmarco_index/', encoder)
corpus = LuceneSearcher.from_prebuilt_index('msmarco-v1-passage')

### Build a HyDE pipeline

In [ ]:
hyde = HyDE(promptor, generator, encoder, searcher)

### Load example Query

In [ ]:
query = 'how long does it take to remove wisdom tooth'

### Build Zeroshot Prompt

In [ ]:
prompt = hyde.prompt(query)
print(prompt)

### Generate Hypothesis Documents

In [ ]:
hypothesis_documents = hyde.generate(query)
for i, doc in enumerate(hypothesis_documents):
    print(f'HyDE Generated Document: {i}')
    print(doc.strip())

### Encode HyDE vector

In [ ]:
hyde_vector = hyde.encode(query, hypothesis_documents)
print(hyde_vector.shape)

### Search Relevant Documents

In [ ]:
hits = hyde.search(hyde_vector, k=10)
for i, hit in enumerate(hits):
    print(f'HyDE Retrieved Document: {i}')
    print(hit.docid)
    print(json.loads(corpus.doc(hit.docid).raw())['contents'])

### End to End Search

e2e search will directly go through all the steps descripted above.

In [ ]:
hits = hyde.e2e_search(query, k=10)
for i, hit in enumerate(hits):
    print(f'HyDE Retrieved Document: {i}')
    print(hit.docid)
    print(json.loads(corpus.doc(hit.docid).raw())['contents'])